# Proyecto Centinela · Fase 1 · Prueba de acoplamiento y test de procedencia

Este cuaderno produce las evidencias de los anexos A y B de la Propuesta de Datos:

1. Descarga RoCoLe v2 desde Mendeley Data (DOI 10.17632/c5yvn32dzg.2) y calcula su huella SHA-256.
2. Reporta la huella de crudeza: filas × columnas, faltantes y balance de la etiqueta.
3. Revisa los metadatos EXIF de las fotografías: fecha de toma y GPS.
4. Descarga el clima mensual de NASA POWER para el sitio de colecta y calcula su huella.
5. Une diez imágenes reales con la serie por la clave (sitio, ventana) e imprime la tabla del anexo A.

Ejecutar en Google Colab con «Entorno de ejecución → Ejecutar todo».

In [ ]:
import hashlib, json, os, zipfile, datetime, glob, io
import requests
import pandas as pd
import numpy as np
from PIL import Image, ExifTags

FECHA_DESCARGA = datetime.date.today().isoformat()
DATA_DIR = "datos"
os.makedirs(DATA_DIR, exist_ok=True)

# Sitio de colecta publicado por los autores (CIIDEA, ESPAM MFL, Calceta, Manabí, Ecuador)
LAT, LON = -0.834206, -80.177159
SITIO = "CIIDEA-ESPAM MFL"
VENTANA = ("2018-05", "2019-04")   # ventana asumida: 12 meses previos a la publicación (2019-05-17)

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

print("Fecha de descarga:", FECHA_DESCARGA)

## 1. Descarga de RoCoLe desde la fuente primaria

La descarga usa la API pública de Mendeley Data asociada al DOI (`/public-api/datasets/c5yvn32dzg/files`, versión 2).
Si la API falla, descarga el .zip a mano con «Download All» desde https://data.mendeley.com/datasets/c5yvn32dzg/2, guárdalo en la carpeta `datos/` y vuelve a ejecutar: el cuaderno usa cualquier .zip que encuentre ahí.

In [ ]:
URL_API_MENDELEY = "https://data.mendeley.com/public-api/datasets/c5yvn32dzg/files"
DIR_ROCOLE = os.path.join(DATA_DIR, "rocole")
os.makedirs(DIR_ROCOLE, exist_ok=True)

zips = sorted(glob.glob(os.path.join(DATA_DIR, "*.zip")))
ARCHIVOS_ROCOLE = []   # (ruta, URL de origen) para el anexo B

if zips:
    # Respaldo: .zip de «Download All» guardado a mano en datos/
    for ZIP_ROCOLE in zips:
        print("Archivo:", os.path.basename(ZIP_ROCOLE),
              "| Tamaño (MB):", round(os.path.getsize(ZIP_ROCOLE) / 1e6, 1))
        print("SHA-256:", sha256(ZIP_ROCOLE))
        ARCHIVOS_ROCOLE.append((ZIP_ROCOLE, "https://doi.org/10.17632/c5yvn32dzg.2"))
        if not glob.glob(os.path.join(DIR_ROCOLE, "**", "*.jpg"), recursive=True):
            with zipfile.ZipFile(ZIP_ROCOLE) as z:
                z.extractall(DIR_ROCOLE)
else:
    r = requests.get(URL_API_MENDELEY, params={"folder_id": "root", "version": 2}, timeout=120)
    r.raise_for_status()
    for a in r.json():
        destino = os.path.join(DIR_ROCOLE, a["filename"])
        if not os.path.exists(destino):
            with requests.get(a["content_details"]["download_url"], stream=True, timeout=600) as d:
                d.raise_for_status()
                with open(destino, "wb") as f:
                    for bloque in d.iter_content(1 << 20):
                        f.write(bloque)
        print("Archivo:", a["filename"], "| SHA-256:", sha256(destino))
        ARCHIVOS_ROCOLE.append((destino, URL_API_MENDELEY))

# Algunos zip de Mendeley traen otro zip adentro: se extrae también (una sola vez)
for interno in glob.glob(os.path.join(DIR_ROCOLE, "**", "*.zip"), recursive=True):
    if not os.path.isdir(os.path.splitext(interno)[0]):
        with zipfile.ZipFile(interno) as z:
            z.extractall(os.path.splitext(interno)[0])

for raiz, dirs, archivos in os.walk(DIR_ROCOLE):
    nivel = raiz.replace(DIR_ROCOLE, "").count(os.sep)
    print("  " * nivel + os.path.basename(raiz) + "/", f"({len(archivos)} archivos)")

## 2. Huella de crudeza (antes de limpiar)

Se buscan los archivos de anotaciones (CSV, XLSX o JSON) y se reportan filas × columnas, faltantes y balance.
Si el cuaderno no identifica la columna de la etiqueta, ajusta `COL_ETIQUETA` con el nombre que muestre `df.columns`.

In [ ]:
anot = [p for p in glob.glob(os.path.join(DIR_ROCOLE, "**", "*"), recursive=True)
        if p.lower().endswith((".csv", ".xlsx", ".xls", ".json"))]
print("Archivos de anotaciones encontrados:")
for p in anot:
    print(" -", os.path.relpath(p, DIR_ROCOLE), "| SHA-256:", sha256(p))

def leer(p):
    if p.lower().endswith(".csv"):
        return pd.read_csv(p)
    if p.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(p)
    with open(p, encoding="utf-8") as f:
        return pd.json_normalize(json.load(f))

tablas = {}
for p in anot:
    try:
        tablas[p] = leer(p)
    except Exception as e:
        print("No se pudo leer", os.path.basename(p), "->", e)

for p, df in tablas.items():
    print("\n==", os.path.basename(p), "| filas × columnas:", df.shape)
    print("Faltantes (%):", round(df.isna().mean().mean() * 100, 2))
    print("Columnas:", list(df.columns)[:30])
    display(df.head(3))

In [ ]:
# Elige la tabla y la columna de etiqueta. Se prefiere la multiclase, porque el anexo B
# reporta los conteos por nivel de roya y ácaro rojo (no solo sana/no sana).
COL_ETIQUETA = None
df_anot = None
candidatas = []
for p, df in tablas.items():
    for c in df.columns:
        valores = df[c].astype(str).str.lower()
        if valores.nunique() <= 20 and valores.str.contains("healthy|rust|red_spider|mite").mean() > 0.5:
            candidatas.append((p, df, c, valores.nunique()))
if candidatas:
    p, df_anot, COL_ETIQUETA, _ = max(candidatas, key=lambda t: t[3])

if COL_ETIQUETA:
    print("Tabla:", os.path.basename(p), "| columna de etiqueta:", COL_ETIQUETA)
    print(df_anot[COL_ETIQUETA].value_counts(dropna=False))
else:
    print("Ajusta COL_ETIQUETA a mano con las columnas impresas arriba.")

## 3. Metadatos EXIF: fecha de toma y GPS

Si las fotografías traen la fecha de toma, la clave puede pasar de agregada (sitio, ventana) a por muestra (sitio, semana).
El GPS se borra antes de subir imágenes a Drive o GitHub (anexo C, bloque F).

In [ ]:
imagenes = sorted(p for p in glob.glob(os.path.join(DIR_ROCOLE, "**", "*"), recursive=True)
                  if p.lower().endswith((".jpg", ".jpeg", ".png")))
print("Imágenes encontradas:", len(imagenes))

TAGS = {v: k for k, v in ExifTags.TAGS.items()}
def exif_info(ruta):
    try:
        with Image.open(ruta) as im:
            ex = im.getexif()
    except Exception:
        return None, False
    if not ex:
        return None, False
    fecha = ex.get_ifd(0x8769).get(TAGS["DateTimeOriginal"]) or ex.get(TAGS["DateTime"])
    gps = bool(ex.get_ifd(0x8825))
    return fecha, gps

# Se revisan todas las imágenes: Pillow solo lee el encabezado, no decodifica la foto
info = [exif_info(p) for p in imagenes]
fecha_por_imagen = {os.path.basename(p): f for p, (f, _) in zip(imagenes, info)}
con_fecha = sum(1 for f, _ in info if f)
con_gps = sum(1 for _, g in info if g)
print(f"De {len(imagenes)} imágenes revisadas: {con_fecha} con fecha EXIF, {con_gps} con GPS EXIF")

fechas = pd.to_datetime(pd.Series([f for f, _ in info if f]), format="%Y:%m:%d %H:%M:%S", errors="coerce").dropna()
if len(fechas):
    print("Rango de fechas EXIF:", fechas.min(), "a", fechas.max())
    print("Imágenes por mes de toma:")
    print(fechas.dt.strftime("%Y-%m").value_counts().sort_index())

## 4. Serie climática de NASA POWER para el sitio

In [ ]:
URL_POWER = "https://power.larc.nasa.gov/api/temporal/monthly/point"
CSV_POWER = os.path.join(DATA_DIR, "nasa_power_ciidea_mensual.csv")
params = dict(parameters="T2M,RH2M,PRECTOTCORR", community="AG",
              latitude=LAT, longitude=LON, start=2018, end=2019, format="JSON")
try:
    r = requests.get(URL_POWER, params=params, timeout=120)
    r.raise_for_status()
    datos = r.json()["properties"]["parameter"]

    clima = (pd.DataFrame(datos)
               .reset_index().rename(columns={"index": "periodo"}))
    clima = clima[~clima["periodo"].str.endswith("13")]          # se quita el promedio anual (mes 13)
    clima["mes"] = pd.to_datetime(clima["periodo"], format="%Y%m").dt.strftime("%Y-%m")
    clima = clima[["mes", "T2M", "RH2M", "PRECTOTCORR"]].reset_index(drop=True)
    clima.to_csv(CSV_POWER, index=False)
    print("Consulta:", r.url)
except requests.RequestException as e:
    # Sin conexión: se usa el CSV descargado antes desde la misma API
    if not os.path.exists(CSV_POWER):
        raise
    print("No se pudo consultar NASA POWER (", e, "); se usa el CSV guardado.")
    clima = pd.read_csv(CSV_POWER, dtype={"mes": str})

print("Archivo:", os.path.basename(CSV_POWER), "| SHA-256:", sha256(CSV_POWER))
display(clima)

## 4b. Serie diaria de NASA POWER para el periodo de toma

Las fotografías traen la fecha de toma en EXIF (abril y mayo de 2019). Con esa fecha, la clave deja de ser agregada y pasa a ser por muestra: (sitio, día de toma).
Se descarga la serie diaria desde 30 días antes de la primera foto, para calcular las condiciones previas a cada toma.

In [ ]:
fechas_toma = pd.to_datetime(pd.Series(fecha_por_imagen), format="%Y:%m:%d %H:%M:%S", errors="coerce")
inicio = (fechas_toma.min() - pd.Timedelta(days=30)).strftime("%Y%m%d")
fin = fechas_toma.max().strftime("%Y%m%d")

URL_POWER_DIARIO = "https://power.larc.nasa.gov/api/temporal/daily/point"
params_d = dict(parameters="T2M,RH2M,PRECTOTCORR", community="AG",
                latitude=LAT, longitude=LON, start=inicio, end=fin, format="JSON")
rd = requests.get(URL_POWER_DIARIO, params=params_d, timeout=120)
rd.raise_for_status()
diario = pd.DataFrame(rd.json()["properties"]["parameter"]).reset_index().rename(columns={"index": "fecha"})
diario["fecha"] = pd.to_datetime(diario["fecha"], format="%Y%m%d")
diario = diario.replace(-999, np.nan)   # -999 = dato faltante en NASA POWER

CSV_POWER_DIARIO = os.path.join(DATA_DIR, "nasa_power_ciidea_diario.csv")
diario.to_csv(CSV_POWER_DIARIO, index=False)
print("Consulta:", rd.url)
print("Archivo:", os.path.basename(CSV_POWER_DIARIO), "| SHA-256:", sha256(CSV_POWER_DIARIO))
print("Días:", len(diario), "| faltantes (%):", round(diario[["T2M", "RH2M", "PRECTOTCORR"]].isna().mean().mean() * 100, 2))

# Humedad relativa media de los 14 días previos a cada fecha (sin incluir el día de toma: no mira el futuro)
diario = diario.sort_values("fecha").set_index("fecha")
diario["RH2M_prev14"] = diario["RH2M"].shift(1).rolling(14).mean()
diario = diario.reset_index()
display(diario.tail())

## 4c. Identificador de planta para la partición agrupada

Los nombres de archivo siguen el patrón `C{n}P{m}...` (p. ej., `C1P2E2.jpg`). Se extrae el prefijo `C{n}P{m}` como identificador de planta y se verifica que haya 390 plantas, como reporta el artículo.

In [ ]:
import re
ids_planta = pd.Series({os.path.basename(p): (re.match(r"^(C\d+P\d+)", os.path.basename(p)) or [None])[0] for p in imagenes})
print("Plantas distintas:", ids_planta.nunique(), "| imágenes sin identificador:", ids_planta.isna().sum())
print("Imágenes por planta:", ids_planta.value_counts().value_counts().sort_index().to_dict())

## 5. Anexo A · Diez filas reales unidas por la clave (sitio, día de toma)

Cada imagen se une con el clima diario de NASA POWER del mismo día de toma y con la humedad relativa media de los 14 días previos.

In [ ]:
def rgb_medio(ruta):
    with Image.open(ruta) as im:
        arr = np.asarray(im.convert("RGB").resize((256, 256)), dtype=float)
    return arr.reshape(-1, 3).mean(axis=0).round(1)

etiqueta = {}
if df_anot is not None and "File" in df_anot.columns:
    etiqueta = dict(zip(df_anot["File"], df_anot[COL_ETIQUETA]))

clima_dia = diario.set_index(diario["fecha"].dt.date)
rng = np.random.default_rng(42)
elegidas = rng.choice(imagenes, size=10, replace=False)

filas = []
for i, p in enumerate(elegidas, 1):
    nombre = os.path.basename(p)
    dia = pd.to_datetime(fecha_por_imagen[nombre], format="%Y:%m:%d %H:%M:%S").date()
    c = clima_dia.loc[dia]
    r_, g_, b_ = rgb_medio(p)
    filas.append({
        "N.º": i,
        "Clave": f"({SITIO}, {dia})",
        "Dato de la rama de imagen": f"{nombre} ({etiqueta.get(nombre, 's/e')}) · RGB medio = ({r_}, {g_}, {b_})",
        "Dato de la serie": (f"T2M = {c['T2M']:.2f} °C · RH2M = {c['RH2M']:.2f} % · "
                             f"PRECTOTCORR = {c['PRECTOTCORR']:.2f} mm/día · RH2M 14 d previos = {c['RH2M_prev14']:.2f} %"),
    })
anexo_a = pd.DataFrame(filas)
display(anexo_a)
print(anexo_a.to_markdown(index=False))   # requiere el paquete tabulate

## 6. Resumen para pegar en los anexos

Copia estas líneas en el anexo B de la Propuesta.

In [ ]:
print("Fecha de descarga:", FECHA_DESCARGA)
for ruta, origen in ARCHIVOS_ROCOLE:
    print(f"| {os.path.basename(ruta)} | {origen} | {FECHA_DESCARGA} | {sha256(ruta)} |")
print(f"| {os.path.basename(CSV_POWER)} | {URL_POWER} | {FECHA_DESCARGA} | {sha256(CSV_POWER)} |")
print(f"| {os.path.basename(CSV_POWER_DIARIO)} | {URL_POWER_DIARIO} | {FECHA_DESCARGA} | {sha256(CSV_POWER_DIARIO)} |")
if df_anot is not None:
    print("Huella de crudeza:", df_anot.shape, "| faltantes (%):", round(df_anot.isna().mean().mean() * 100, 2))
    print("Balance de la etiqueta:", df_anot[COL_ETIQUETA].value_counts().to_dict())
print("Imágenes:", len(imagenes))